In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import json
import os
import glob
import gc

import numpy as np
import torch as th
import matplotlib.pyplot as plt
from pathlib import Path

# Repo root is three directories above this notebook (notebooks/lqr/inputs/ -> ctrlwam/)
REPO_ROOT = Path(".").resolve().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [ ]:
DATA_DIR = "policy_inputs_new_cam"
OUTPUT_DIR = "cam_random_large"
ACTIVATIONS_PATH = "activations_dict_new_cam.npz"

# Tasks to analyse — each must have at least one positive.npz and one negative.npz
TASKS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
# TASKS = [1, 2, 9, 7]


def get_task_dirs(task_id):
    # return sorted(glob.glob(os.path.join(DATA_DIR, f"libero_10__task{task_id:02d}__gripper_perturb*")))
    # return sorted(glob.glob(os.path.join(DATA_DIR, f"libero_10__task{task_id:02d}__cam_perturb*")))
    return sorted(glob.glob(os.path.join(DATA_DIR, f"libero_10__task{task_id:02d}_*")))


def get_task_files(task_id):
    dirs = get_task_dirs(task_id)
    pos = [os.path.join(d, "positive.npz") for d in dirs if os.path.exists(os.path.join(d, "positive.npz"))]
    neg = [os.path.join(d, "negative.npz") for d in dirs if os.path.exists(os.path.join(d, "negative.npz"))]
    return pos, neg


def get_task_language(task_id):
    for d in get_task_dirs(task_id):
        manifest = os.path.join(d, "manifest.json")
        if os.path.exists(manifest):
            with open(manifest) as f:
                return json.load(f)["policy_prompt"]
                # return json.load(f)["task_language"]
    raise FileNotFoundError(f"No manifest.json found for task {task_id}")


# Validate upfront — error immediately if a task is missing positive or negative data
for t in TASKS:
    pos, neg = get_task_files(t)
    if not pos:
        raise FileNotFoundError(f"Task {t}: no positive.npz files found under {DATA_DIR}/")
    if not neg:
        raise FileNotFoundError(f"Task {t}: no negative.npz files found under {DATA_DIR}/")
    lang = get_task_language(t)
    print(f"Task {t:2d}: {len(pos)} pos, {len(neg)} neg  —  {lang!r}")

In [ ]:
from cosmos_policy.experiments.robot.libero.run_libero_eval import PolicyEvalConfig
from cosmos_policy.experiments.robot.cosmos_utils import (
    get_action,
    get_model,
    load_dataset_stats,
    init_t5_text_embeddings_cache,
)

cfg = PolicyEvalConfig(
    config="cosmos_predict2_2b_480p_libero__inference_only",
    ckpt_path="nvidia/Cosmos-Policy-LIBERO-Predict2-2B",
    config_file="cosmos_policy/config/config.py",
    dataset_stats_path="nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_dataset_statistics.json",
    t5_text_embeddings_path="nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_t5_embeddings.pkl",
    use_wrist_image=True,
    use_proprio=True,
    normalize_proprio=True,
    unnormalize_actions=True,
    chunk_size=16,
    num_open_loop_steps=16,
    trained_with_image_aug=True,
    use_jpeg_compression=True,
    flip_images=True,
    num_denoising_steps_action=1,  # single denoising step — sufficient for activation probing
    num_denoising_steps_future_state=1,
    num_denoising_steps_value=1,
    task_suite_name="libero_10",
    suite="libero",
)
dataset_stats = load_dataset_stats(cfg.dataset_stats_path)
init_t5_text_embeddings_cache(cfg.t5_text_embeddings_path)
model, _ = get_model(cfg)

N_BLOCKS = len(model.net.blocks)
print(f"Model loaded — {N_BLOCKS} DiT blocks")

In [ ]:
# Collect per-block activations for every observation row in each task's npz files.
# Forward hooks on model.net.blocks capture the block output (x_B_T_H_W_D),
# mean-pooled over spatial/temporal dims to give a [D]-dim vector per block.

def collect_activations_from_npz(npz_path, task_language):
    """
    Run each observation row through one denoising step with hooks active.
    Returns a list of dicts {block_idx: np.ndarray[D]}, one dict per row.
    """
    d = np.load(npz_path)
    primary_images = d["primary_images"]   # [N, H, W, C] uint8
    wrist_images   = d["wrist_images"]     # [N, H, W, C] uint8
    proprios       = d["proprios"]         # [N, 9] float32

    _buf = {}
    hooks = []
    for b in range(N_BLOCKS):
        def _hook(module, inp, out, _b=b):
            # out: [B, T, H, W, D] — mean-pool all non-channel dims → [D]
            _buf[_b] = out.detach().float().mean(dim=(0, 1, 2, 3)).cpu().numpy()
        hooks.append(model.net.blocks[b].register_forward_hook(_hook))

    rows = []
    for i in range(len(primary_images)):
        _buf.clear()
        obs = {
            "primary_image": primary_images[i],
            "wrist_image":   wrist_images[i],
            "proprio":       proprios[i],
        }
        get_action(cfg, model, dataset_stats, obs, task_language,
                   num_denoising_steps_action=1)
        rows.append({b: _buf[b].copy() for b in range(N_BLOCKS)})

    for h in hooks:
        h.remove()
    return rows


def collect_full_activations_from_npz(npz_path, task_language):
    """
    Run each observation row through one denoising step with hooks active.
    Returns a list of dicts {block_idx: np.ndarray[D]}, one dict per row.
    """
    d = np.load(npz_path)
    primary_images = d["primary_images"]   # [N, H, W, C] uint8
    wrist_images   = d["wrist_images"]     # [N, H, W, C] uint8
    proprios       = d["proprios"]         # [N, 9] float32

    _buf = {}
    hooks = []
    for b in range(N_BLOCKS):
        def _hook(module, inp, out, _b=b):
            # out: [B, T, H, W, D] — mean-pool all non-channel dims → [D]
            _buf[_b] = out.detach().float().mean(dim=(0, 1, 2, 3)).cpu().numpy()
        hooks.append(model.net.blocks[b].register_forward_hook(_hook))

    rows = []
    for i in range(len(primary_images)):
        _buf.clear()
        obs = {
            "primary_image": primary_images[i],
            "wrist_image":   wrist_images[i],
            "proprio":       proprios[i],
        }
        get_action(cfg, model, dataset_stats, obs, task_language,
                   num_denoising_steps_action=1)
        rows.append({b: _buf[b].copy() for b in range(N_BLOCKS)})

    for h in hooks:
        h.remove()
    return rows



# activations_by_task[t] = {"positive": [...rows...], "negative": [...rows...]}
filepath = Path(ACTIVATIONS_PATH)
if filepath.is_file():
    activations_by_task = np.load(filepath, allow_pickle=True)
    activations_by_task = {int(k): v for k, v in activations_by_task.items()}
else:
    activations_by_task = {}

print(activations_by_task)

for t in TASKS:
    if t not in activations_by_task:
        lang = get_task_language(t)
        pos_files, neg_files = get_task_files(t)

        n_pos = sum(np.load(f)["primary_images"].shape[0] for f in pos_files)
        n_neg = sum(np.load(f)["primary_images"].shape[0] for f in neg_files)
        print(f"Task {t}: collecting {n_pos} pos + {n_neg} neg rows...")

        pos_rows = []
        for f in pos_files:
            pos_rows.extend(collect_activations_from_npz(f, lang))
            print(f"  pos: {len(pos_rows)} rows collected", end="\r")

        neg_rows = []
        for f in neg_files:
            neg_rows.extend(collect_activations_from_npz(f, lang))
            print(f"  neg: {len(neg_rows)} rows collected", end="\r")

        activations_by_task[t] = {"positive": pos_rows, "negative": neg_rows}
        print(f"Task {t}: done — {len(pos_rows)} positive, {len(neg_rows)} negative observations")

print("\nActivation collection complete")
str_keyed_dict = {str(k): v for k, v in activations_by_task.items()}

np.savez(ACTIVATIONS_PATH, **str_keyed_dict)

In [ ]:
# Load pre-computed activations from disk instead of running inference.
# Run this cell instead of the model-loading and collection cells above.
#
# Saved format: npz with string task keys, each value a 0-d object array
# wrapping {label: [list of {block_int: ndarray[D]}]}.
# This cell converts that into the activations_by_task dict the rest of the
# notebook expects: {task_int: {"positive": [...], "negative": [...]}}.


raw = np.load(ACTIVATIONS_PATH, allow_pickle=True)
available_tasks = {int(k) for k in raw.keys()}
missing = set(TASKS) - available_tasks
if missing:
    raise FileNotFoundError(
        f"Tasks {sorted(missing)} not found in {ACTIVATIONS_PATH}. "
        f"Available: {sorted(available_tasks)}"
    )

activations_by_task = {int(k): raw[k].item() for k in raw.keys() if int(k) in set(TASKS)}

# Derive N_BLOCKS from the loaded data
_sample_task = next(iter(activations_by_task.values()))
_sample_rows = _sample_task["positive"] or _sample_task["negative"]
N_BLOCKS = len(_sample_rows[0])

print(f"Loaded activations for tasks {sorted(activations_by_task)} from {ACTIVATIONS_PATH!r}")
for t in TASKS:
    n_pos = len(activations_by_task[t]["positive"])
    n_neg = len(activations_by_task[t]["negative"])
    print(f"  Task {t:2d}: {n_pos} positive, {n_neg} negative rows, {N_BLOCKS} blocks")

In [ ]:
# filepath = Path("activations_dict.npz")
# if filepath.is_file():
#     activations_by_task_test = np.load(filepath, allow_pickle=True)
#     activations_by_task_test = {int(k): v for k, v in activations_by_task_test.items()}
# else:
#     activations_by_task_test = {}

# print(activations_by_task_test)


In [ ]:
# str_keyed_dict = {str(k): v for k, v in activations_by_task.items()}

# np.savez('activations_dict.npz', **str_keyed_dict)

In [ ]:
# Fit contrastive PCA per block per task.
# Uses gram matrix trick (n_samples x n_samples) since n_samples << n_features.
# Contrastive vector for pair i at block b = pos_rows[i][b] - neg_rows[i][b].

pca_by_task_block = {}
n_pairs_by_task = {}

for t in TASKS:
    pos_rows = activations_by_task[t]["positive"]
    neg_rows = activations_by_task[t]["negative"]
    n_pairs = min(len(pos_rows), len(neg_rows))
    n_pairs_by_task[t] = n_pairs

    if n_pairs < 3:
        raise ValueError(f"Task {t}: only {n_pairs} contrastive pair(s) — need at least 3 for PCA")

    pca_by_task_block[t] = {}
    for b in range(N_BLOCKS):
        X = np.stack([pos_rows[i][b] - neg_rows[i][b] for i in range(n_pairs)])  # [n_pairs, D]
        mean = X.mean(axis=0)
        X_c = X - mean
        G = X_c @ X_c.T
        eigenvalues, eigenvectors = np.linalg.eigh(G)
        top_vals = eigenvalues[-3:][::-1].copy()
        top_vecs = eigenvectors[:, -3:][:, ::-1].copy()
        components = X_c.T @ top_vecs / np.sqrt(top_vals)
        pca_by_task_block[t][b] = {"mean": mean, "components": components}
        del X, X_c, G

    gc.collect()
    D = pca_by_task_block[t][0]["components"].shape[0]
    print(f"Task {t}: PCA fitted — {N_BLOCKS} blocks, {n_pairs} pairs, feature dim {D}")

print("PCA done")

In [ ]:
# Project all positive and negative activations onto per-block PCA planes per task.

projs_by_task_block = {}

for t in TASKS:
    pos_rows = activations_by_task[t]["positive"]
    neg_rows = activations_by_task[t]["negative"]

    projs_by_task_block[t] = {}
    for b in range(N_BLOCKS):
        mean  = pca_by_task_block[t][b]["mean"]
        comps = pca_by_task_block[t][b]["components"]
        pos_projs = np.stack([(r[b] - mean) @ comps for r in pos_rows])
        neg_projs = np.stack([(r[b] - mean) @ comps for r in neg_rows])
        projs_by_task_block[t][b] = {"positive": pos_projs, "negative": neg_projs}

    print(f"Task {t}: projected {len(pos_rows)} pos, {len(neg_rows)} neg onto {N_BLOCKS} block planes")

print("Projections done")

In [ ]:
from scipy.optimize import minimize


losses = []
def fit_svm(pos_pts, neg_pts, C=10.0):
    """Soft-margin linear SVM via L-BFGS-B."""
    X = np.vstack([pos_pts, neg_pts])
    y = np.concatenate([np.ones(len(pos_pts)), -np.ones(len(neg_pts))])
    d = X.shape[1]
    # print(f"d in SVM: {d}")

    losses.clear()
    def obj_and_grad(params):
        w, b = params[:d], params[d]
        margins = y * (X @ w + b)
        mask = margins < 1
        loss = 0.5 * np.dot(w, w) + C * np.maximum(0, 1 - margins).sum()
        grad_w = w - C * (y[mask, None] * X[mask]).sum(axis=0)
        grad_b = float(-C * y[mask].sum())
        
        # print(f"regularization: {np.dot(w, w)}")
        losses.append(loss - 0.5 * np.dot(w, w))
        return loss, np.concatenate([grad_w, [grad_b]])

    res = minimize(obj_and_grad, np.zeros(d + 1), jac=True, method="L-BFGS-B")
    # print(f"Final loss: {losses[-1]}")
    return res.x[:d], res.x[d], losses[-1]


def plot_hyperplane_2d(ax, w, bias, all_pts):
    xlim = all_pts[:, 0].min(), all_pts[:, 0].max()
    ylim = all_pts[:, 1].min(), all_pts[:, 1].max()
    xs = np.linspace(xlim[0], xlim[1], 300)
    if abs(w[1]) > 1e-12:
        ys = -(w[0] * xs + bias) / w[1]
    else:
        ax.axvline(-bias / w[0], color="k", ls="--", lw=0.8, alpha=0.6)
        return
    ax.plot(xs, ys, color="k", ls="--", lw=0.8, alpha=0.6)
    px, py = 0.05 * (xlim[1] - xlim[0]), 0.05 * (ylim[1] - ylim[0])
    ax.set_xlim(xlim[0] - px, xlim[1] + px)
    ax.set_ylim(ylim[0] - py, ylim[1] + py)


def plot_hyperplane_3d(ax, w, bias, all_pts):
    i = int(np.argmax(np.abs(w)))
    j, k = [d for d in range(3) if d != i]
    JJ, KK = np.meshgrid(
        np.linspace(all_pts[:, j].min(), all_pts[:, j].max(), 15),
        np.linspace(all_pts[:, k].min(), all_pts[:, k].max(), 15),
    )
    II = -(w[j] * JJ + w[k] * KK + bias) / w[i]
    II = np.clip(II, all_pts[:, i].min(), all_pts[:, i].max())
    coords = [None, None, None]
    coords[i], coords[j], coords[k] = II, JJ, KK
    ax.plot_surface(*coords, alpha=0.18, color="gray", linewidth=0, antialiased=False)


svm_by_task_block = {}
svm_losses_by_task_block = {}

processed = {}
for t in TASKS:
    # print(f"Running task {t}")
    svm_by_task_block[t] = {}
    svm_losses_by_task_block[t] = {}
    pos_pts = projs_by_task_block[t][0]["positive"]
    neg_pts = projs_by_task_block[t][0]["negative"]
    n = len(pos_pts) + len(neg_pts)
    processed[t] = {}
    processed[t]["num_samples"] = n
    losses_per_task = []
    for b in range(N_BLOCKS):
        # print(f"Running block {b}")
        pos_pts = projs_by_task_block[t][b]["positive"]
        neg_pts = projs_by_task_block[t][b]["negative"]
        # print(len(pos_pts))

        if len(pos_pts) > 1 and len(neg_pts) > 1:
            w, bias, loss = fit_svm(pos_pts, neg_pts)
            svm_by_task_block[t][b] = {"w": w, "bias": bias}
            svm_losses_by_task_block[t][b] = loss
            losses_per_task.append(loss)
        else:
            svm_by_task_block[t][b] = None
    processed[t]["best_loss"] = min(losses_per_task)
    processed[t]["last_loss"] = losses_per_task[-1]
    processed[t]["best_avg_loss"] = (processed[t]["best_loss"] / 10.0) / n
    processed[t]["last_avg_loss"] = (losses_per_task[-1] / 10.0) / n
    print(f"  task {t} SVMs fitted", end="\r")

print(f"\nSVMs fitted for all {len(TASKS)} tasks × {N_BLOCKS} blocks")
# print(svm_losses_by_task_block)

import pandas as pd

df = pd.DataFrame.from_dict(svm_losses_by_task_block, orient="index")

# Optional: convert numpy floats
df = df.astype(float)

print("Raw data")
print(df.to_markdown())

df = pd.DataFrame.from_dict(processed, orient="index")

# Optional: convert numpy floats
df = df.astype(float)

print("Processed")
print(df.to_markdown())

In [ ]:
# 2D scatter — one figure per task, one subplot per DiT block
N_COLS = 6
N_ROWS = (N_BLOCKS + N_COLS - 1) // N_COLS

COLORS = {"positive": "#2196F3", "negative": "#FF5722"}

for t in TASKS:
    lang = get_task_language(t)
    n_pairs = n_pairs_by_task[t]
    n_pos = len(activations_by_task[t]["positive"])
    n_neg = len(activations_by_task[t]["negative"])

    fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(N_COLS * 3.5, N_ROWS * 3.2))
    axes = axes.flatten()

    for b in range(N_BLOCKS):
        ax = axes[b]
        pos_pts = projs_by_task_block[t][b]["positive"]
        neg_pts = projs_by_task_block[t][b]["negative"]
        all_pts = np.vstack([pos_pts, neg_pts])

        for label, color in COLORS.items():
            pts = projs_by_task_block[t][b][label]
            ax.scatter(pts[:, 0], pts[:, 1], c=color, alpha=0.35, s=6, label=label, rasterized=True)

        if svm_by_task_block[t][b] is not None:
            plot_hyperplane_2d(ax, svm_by_task_block[t][b]["w"], svm_by_task_block[t][b]["bias"], all_pts)

        ax.set_title(f"Block {b}", fontsize=8, pad=2)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel("PC1", fontsize=6, labelpad=1)
        ax.set_ylabel("PC2", fontsize=6, labelpad=1)

    for b in range(N_BLOCKS, len(axes)):
        axes[b].set_visible(False)

    handles, labels_ = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels_, loc="lower right", fontsize=11, markerscale=4,
               title="Condition", title_fontsize=10)

    fig.suptitle(
        f"Task {t} — DiT block activations projected onto contrastive PCA planes\n"
        f"{lang!r}\n"
        f"(PCA fit on {n_pairs} pairs; {n_pos} positive, {n_neg} negative projected)",
        fontsize=10,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    out_path = Path(f"{OUTPUT_DIR}/cosmos_contrastive_pca_2d_task{t:02d}.png")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

for t in TASKS:
    lang = get_task_language(t)
    n_pairs = n_pairs_by_task[t]
    n_pos = len(activations_by_task[t]["positive"])
    n_neg = len(activations_by_task[t]["negative"])

    fig = plt.figure(figsize=(N_COLS * 3.5, N_ROWS * 3.2))

    for b in range(N_BLOCKS):
        ax = fig.add_subplot(N_ROWS, N_COLS, b + 1, projection="3d")
        pos_pts = projs_by_task_block[t][b]["positive"]
        neg_pts = projs_by_task_block[t][b]["negative"]
        all_pts = np.vstack([pos_pts, neg_pts])

        if svm_by_task_block[t][b] is not None:
            plot_hyperplane_3d(ax, svm_by_task_block[t][b]["w"], svm_by_task_block[t][b]["bias"], all_pts)

        for label, color in COLORS.items():
            pts = projs_by_task_block[t][b][label]
            ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2],
                       c=color, alpha=0.35, s=4, label=label, rasterized=True)

        ax.set_title(f"Block {b}", fontsize=8, pad=2)
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])
        ax.set_xlabel("PC1", fontsize=5, labelpad=-8)
        ax.set_ylabel("PC2", fontsize=5, labelpad=-8)
        ax.set_zlabel("PC3", fontsize=5, labelpad=-8)

    handles, labels_ = fig.axes[0].get_legend_handles_labels()
    fig.legend(handles, labels_, loc="lower right", fontsize=11, markerscale=4,
               title="Condition", title_fontsize=10)

    fig.suptitle(
        f"Task {t} — DiT block activations, top 3 contrastive PCs (3D)\n"
        f"{lang!r}\n"
        f"(PCA fit on {n_pairs} pairs; {n_pos} positive, {n_neg} negative projected)",
        fontsize=10,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    out_path = f"{OUTPUT_DIR}/cosmos_contrastive_pca_3d_task{t:02d}.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {out_path}")

In [ ]:
import itertools
import pandas as pd

PCS = [1, 2]  # which PCs to plot (1-indexed); e.g. [3, 1] → x=PC3, y=PC1

assert len(PCS) == 2, "PCS must have exactly 2 entries"

COLORS  = {"positive": "#2196F3", "negative": "#FF5722"}
MARKERS = ['o', 's', '^', 'D', 'v', 'p', '*', 'h']

n_pcs_needed = max(max(PCS), 3)  # always compute at least 3 for the 3D SVM table
pc_cols = [p - 1 for p in PCS]  # 0-indexed column selection into sorted-desc components

N_COLS = 6
N_ROWS = (N_BLOCKS + N_COLS - 1) // N_COLS

# avg hinge loss per sample, keyed by (t1, t2) then block index
pair_losses    = {}
pair_losses_3d = {}

for COMBINED_TASKS in itertools.combinations(TASKS, 2):
    COMBINED_TASKS = list(COMBINED_TASKS)
    pair_key = tuple(COMBINED_TASKS)

    missing = set(COMBINED_TASKS) - set(activations_by_task)
    if missing:
        raise KeyError(f"Tasks {sorted(missing)} not in activations_by_task.")

    TASK_MARKER = {t: MARKERS[i % len(MARKERS)] for i, t in enumerate(COMBINED_TASKS)}

    # ── combined PCA per block ────────────────────────────────────────────────
    combined_pca = {}
    for b in range(N_BLOCKS):
        vecs = []
        for t in COMBINED_TASKS:
            pos_rows = activations_by_task[t]["positive"]
            neg_rows = activations_by_task[t]["negative"]
            n_pairs = min(len(pos_rows), len(neg_rows))
            vecs.extend(pos_rows[i][b] - neg_rows[i][b] for i in range(n_pairs))

        X = np.stack(vecs)
        mean = X.mean(axis=0)
        X_c = X - mean
        G = X_c @ X_c.T
        eigenvalues, eigenvectors = np.linalg.eigh(G)
        n_keep = min(n_pcs_needed, len(eigenvalues))
        top_vals = eigenvalues[-n_keep:][::-1].copy()
        top_vecs = eigenvectors[:, -n_keep:][:, ::-1].copy()
        all_components = X_c.T @ top_vecs / np.sqrt(top_vals)
        combined_pca[b] = {
            "mean": mean,
            "components":    all_components[:, pc_cols],  # [D, 2] for plotting
            "components_3d": all_components[:, :3],        # [D, 3] for 3D SVM
        }

    # ── project all rows ──────────────────────────────────────────────────────
    combined_projs    = {}
    combined_projs_3d = {}
    for t in COMBINED_TASKS:
        combined_projs[t]    = {}
        combined_projs_3d[t] = {}
        for label in ("positive", "negative"):
            rows = activations_by_task[t][label]
            combined_projs[t][label] = np.stack(
                [np.stack([(r[b] - combined_pca[b]["mean"]) @ combined_pca[b]["components"]
                           for b in range(N_BLOCKS)])
                 for r in rows]
            )  # [N_rows, N_BLOCKS, 2]
            combined_projs_3d[t][label] = np.stack(
                [np.stack([(r[b] - combined_pca[b]["mean"]) @ combined_pca[b]["components_3d"]
                           for b in range(N_BLOCKS)])
                 for r in rows]
            )  # [N_rows, N_BLOCKS, 3]

    # ── SVM + loss collection ─────────────────────────────────────────────────
    combined_svm = {}
    block_losses    = {}
    block_losses_3d = {}
    for b in range(N_BLOCKS):
        all_pos    = np.vstack([combined_projs[t]["positive"][:, b, :]    for t in COMBINED_TASKS])
        all_neg    = np.vstack([combined_projs[t]["negative"][:, b, :]    for t in COMBINED_TASKS])
        all_pos_3d = np.vstack([combined_projs_3d[t]["positive"][:, b, :] for t in COMBINED_TASKS])
        all_neg_3d = np.vstack([combined_projs_3d[t]["negative"][:, b, :] for t in COMBINED_TASKS])
        n_total = len(all_pos) + len(all_neg)
        if len(all_pos) > 1 and len(all_neg) > 1:
            w, bias, hinge = fit_svm(all_pos, all_neg)
            combined_svm[b] = {"w": w, "bias": bias}
            block_losses[b] = hinge / n_total / 10

            _, _, hinge_3d = fit_svm(all_pos_3d, all_neg_3d)
            block_losses_3d[b] = hinge_3d / n_total / 10
        else:
            combined_svm[b] = None
            block_losses[b]    = float("nan")
            block_losses_3d[b] = float("nan")

    pair_losses[pair_key]    = block_losses
    pair_losses_3d[pair_key] = block_losses_3d

    # ── plot ──────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(N_ROWS, N_COLS, figsize=(N_COLS * 3.5, N_ROWS * 3.2))
    axes = axes.flatten()

    for b in range(N_BLOCKS):
        ax = axes[b]
        all_pts_list = []
        for t in COMBINED_TASKS:
            for label in ("positive", "negative"):
                pts = combined_projs[t][label][:, b, :]
                all_pts_list.append(pts)
                ax.scatter(pts[:, 0], pts[:, 1],
                           c=COLORS[label], marker=TASK_MARKER[t],
                           alpha=0.35, s=6, label=f"task {t} {label}", rasterized=True)
        all_pts = np.vstack(all_pts_list)
        if combined_svm[b] is not None:
            plot_hyperplane_2d(ax, combined_svm[b]["w"], combined_svm[b]["bias"], all_pts)
        ax.set_title(f"Block {b}", fontsize=8, pad=2)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(f"PC{PCS[0]}", fontsize=6, labelpad=1)
        ax.set_ylabel(f"PC{PCS[1]}", fontsize=6, labelpad=1)

    for b in range(N_BLOCKS, len(axes)):
        axes[b].set_visible(False)

    handles, labels_ = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels_, loc="lower right", fontsize=9, markerscale=2,
               title=f"tasks {COMBINED_TASKS}", title_fontsize=9, ncol=len(COMBINED_TASKS))

    n_pairs_total = sum(min(len(activations_by_task[t]["positive"]),
                            len(activations_by_task[t]["negative"])) for t in COMBINED_TASKS)
    fig.suptitle(
        f"Combined tasks {COMBINED_TASKS} — DiT block activations  (PC{PCS[0]} vs PC{PCS[1]})\n"
        f"(PCA fit on {n_pairs_total} pooled contrastive pairs; color=condition  shape=task)",
        fontsize=10,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    out_path = Path(f"{OUTPUT_DIR}/cosmos_combined_tasks_{'_'.join(str(t) for t in COMBINED_TASKS)}_pc{PCS[0]}v{PCS[1]}_2d.png")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    # plt.show()
    print(f"Saved → {out_path}")

# ── hinge loss summary tables ─────────────────────────────────────────────────
def _loss_summary(losses_dict, label):
    df = pd.DataFrame.from_dict(losses_dict, orient="index").astype(float)
    df.index = [str(p) for p in df.index]
    df.columns = [f"b{b}" for b in df.columns]
    summary = pd.DataFrame({
        "best block":            df.idxmin(axis=1),
        "best avg loss":         df.min(axis=1),
        f"b{N_BLOCKS-1} avg loss": df[f"b{N_BLOCKS-1}"],
    })
    print(f"\n### Avg hinge loss per sample — all task pairs  [{label}]\n")
    print(df.to_markdown(floatfmt=".4f"))
    print(f"\n### Summary: best block and last block  [{label}]\n")
    print(summary.to_markdown(floatfmt=".4f"))

_loss_summary(pair_losses,    "2D SVM — top-2 PCs")
_loss_summary(pair_losses_3d, "3D SVM — top-3 PCs")

In [ ]:
# PC sweep (combined tasks): fit one contrastive PCA per block pooling all
# SWEEP_TASKS, then for every unique PC pair (PC_i vs PC_j, i < j, i,j in 1..N_PCS)
# produce one figure with one subplot per DiT block — all tasks overlaid,
# shape=task, color=condition.  Mirrors the combined-tasks cell above.

import itertools

N_PCS       = 2      # number of PCs to compute; plots all C(N_PCS, 2) pairs
SWEEP_TASKS = TASKS  # subset of TASKS to overlay, e.g. [1, 5, 7]

_S_COLORS  = {"positive": "#2196F3", "negative": "#FF5722"}
_S_MARKERS = ['o', 's', '^', 'D', 'v', 'p', '*', 'h']
_S_N_COLS  = 6
_S_N_ROWS  = (N_BLOCKS + _S_N_COLS - 1) // _S_N_COLS

TASK_MARKER = {t: _S_MARKERS[i % len(_S_MARKERS)] for i, t in enumerate(SWEEP_TASKS)}

# ── 1. Fit combined contrastive PCA with N_PCS components per block ───────────
combined_pca_sweep = {}
for b in range(N_BLOCKS):
    vecs = []
    for t in SWEEP_TASKS:
        pos_rows = activations_by_task[t]["positive"]
        neg_rows = activations_by_task[t]["negative"]
        n_pairs  = min(len(pos_rows), len(neg_rows))
        vecs.extend(pos_rows[i][b] - neg_rows[i][b] for i in range(n_pairs))

    X      = np.stack(vecs)
    mean   = X.mean(axis=0)
    X_c    = X - mean
    n_keep = min(N_PCS, len(vecs) - 1)
    G      = X_c @ X_c.T
    vals, evecs = np.linalg.eigh(G)
    top_vals = vals[-n_keep:][::-1].copy()
    top_vecs = evecs[:, -n_keep:][:, ::-1].copy()
    comps  = X_c.T @ top_vecs / np.sqrt(top_vals)   # [D, n_keep]
    combined_pca_sweep[b] = {"mean": mean, "components": comps, "n_keep": n_keep}

total_pairs = sum(min(len(activations_by_task[t]["positive"]),
                      len(activations_by_task[t]["negative"])) for t in SWEEP_TASKS)
n_keep = combined_pca_sweep[0]["n_keep"]
pc_pairs = list(itertools.combinations(range(1, n_keep + 1), 2))  # 1-indexed
print(f"Combined sweep PCA: {N_BLOCKS} blocks, {n_keep} PCs, {total_pairs} pooled pairs")
print(f"Plotting {len(pc_pairs)} PC pairs: {pc_pairs}")

# ── 2. Project each task's activations onto the shared PCA basis ──────────────
projs_sweep = {}
for t in SWEEP_TASKS:
    projs_sweep[t] = {}
    for b in range(N_BLOCKS):
        mean  = combined_pca_sweep[b]["mean"]
        comps = combined_pca_sweep[b]["components"]
        pos_p = np.stack([(r[b] - mean) @ comps for r in activations_by_task[t]["positive"]])
        neg_p = np.stack([(r[b] - mean) @ comps for r in activations_by_task[t]["negative"]])
        projs_sweep[t][b] = {"positive": pos_p, "negative": neg_p}

# ── 3. Plot every PC pair — one figure per pair, all tasks overlaid ───────────
for pc_a, pc_b in pc_pairs:
    col_a, col_b = pc_a - 1, pc_b - 1   # 0-indexed

    fig, axes = plt.subplots(_S_N_ROWS, _S_N_COLS,
                             figsize=(_S_N_COLS * 3.2, _S_N_ROWS * 3.0))
    axes = axes.flatten()

    for b in range(N_BLOCKS):
        ax = axes[b]
        for t in SWEEP_TASKS:
            for label in ("positive", "negative"):
                pts = projs_sweep[t][b][label]
                ax.scatter(pts[:, col_a], pts[:, col_b],
                           c=_S_COLORS[label], marker=TASK_MARKER[t],
                           alpha=0.35, s=6, label=f"task {t} {label}",
                           rasterized=True)
        ax.set_title(f"Block {b}", fontsize=8, pad=2)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(f"PC{pc_a}", fontsize=6, labelpad=1)
        ax.set_ylabel(f"PC{pc_b}", fontsize=6, labelpad=1)

    for b in range(N_BLOCKS, len(axes)):
        axes[b].set_visible(False)

    handles, lbls = axes[0].get_legend_handles_labels()
    fig.legend(handles, lbls, loc="lower right", fontsize=9, markerscale=2,
               title=f"tasks {SWEEP_TASKS}", title_fontsize=9,
               ncol=len(SWEEP_TASKS))
    fig.suptitle(
        f"Combined tasks {SWEEP_TASKS} — PC{pc_a} vs PC{pc_b}  ({N_BLOCKS} DiT blocks)\n"
        f"(PCA fit on {total_pairs} pooled contrastive pairs; color=condition  shape=task)",
        fontsize=10,
    )
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    out_path = Path(f"{OUTPUT_DIR}/cosmos_pc_sweep_combined_pc{pc_a}_vs_pc{pc_b}.png")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    # plt.show()
    print(f"Saved → {out_path}")

In [ ]:
# Layer-pooled PCA: group DiT blocks into consecutive windows of size K, pool
# all contrastive vectors (across tasks AND blocks within each window), fit PCA,
# then project every (observation, block) pair onto PC1 vs PC2.
# One subplot per window; all tasks overlaid — color=condition, shape=task.

K          = 3      # window size in blocks; windows are [0,K), [K,2K), …
POOL_TASKS = [1,7]  # subset of TASKS to overlay, e.g. [1, 5, 7]

_L_COLORS  = {"positive": "#2196F3", "negative": "#FF5722"}
_L_MARKERS = ['o', 's', '^', 'D', 'v', 'p', '*', 'h']
_L_TASK_MARKER = {t: _L_MARKERS[i % len(_L_MARKERS)] for i, t in enumerate(POOL_TASKS)}

windows = [(s, min(s + K, N_BLOCKS)) for s in range(0, N_BLOCKS, K)]
n_windows = len(windows)

# ── 1. Fit PCA per window (pool contrastive vectors over tasks + blocks) ──────
pca_layer = {}
for w_start, w_end in windows:
    vecs = []
    for t in POOL_TASKS:
        pos_rows = activations_by_task[t]["positive"]
        neg_rows = activations_by_task[t]["negative"]
        n_pairs  = min(len(pos_rows), len(neg_rows))
        for b in range(w_start, w_end):
            vecs.extend(pos_rows[i][b] - neg_rows[i][b] for i in range(n_pairs))

    X     = np.stack(vecs)
    mean  = X.mean(axis=0)
    X_c   = X - mean
    G     = X_c @ X_c.T
    vals, evecs = np.linalg.eigh(G)
    top_vals = vals[-2:][::-1].copy()
    top_vecs = evecs[:, -2:][:, ::-1].copy()
    comps = X_c.T @ top_vecs / np.sqrt(top_vals)   # [D, 2]
    pca_layer[(w_start, w_end)] = {"mean": mean, "components": comps}
    print(f"Window blocks [{w_start:2d}, {w_end:2d}): PCA fitted on {len(vecs)} vectors")

# ── 2. Project every (observation, block) pair onto each window's PC plane ────
projs_layer = {}   # projs_layer[(w_start,w_end)][t][label] → [N_obs * (w_end-w_start), 2]
for w_start, w_end in windows:
    mean  = pca_layer[(w_start, w_end)]["mean"]
    comps = pca_layer[(w_start, w_end)]["components"]
    projs_layer[(w_start, w_end)] = {}
    for t in POOL_TASKS:
        projs_layer[(w_start, w_end)][t] = {}
        for label in ("positive", "negative"):
            rows = activations_by_task[t][label]
            pts  = np.vstack([
                (r[b] - mean) @ comps
                for r in rows
                for b in range(w_start, w_end)
            ])
            projs_layer[(w_start, w_end)][t][label] = pts

# ── 3. Plot — one subplot per window ─────────────────────────────────────────
n_cols = min(n_windows, 4)
n_rows = (n_windows + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4.0, n_rows * 3.8),
                          squeeze=False)
axes = axes.flatten()

for idx, (w_start, w_end) in enumerate(windows):
    ax = axes[idx]
    for t in POOL_TASKS:
        for label in ("positive", "negative"):
            pts = projs_layer[(w_start, w_end)][t][label]
            ax.scatter(pts[:, 0], pts[:, 1],
                       c=_L_COLORS[label], marker=_L_TASK_MARKER[t],
                       alpha=0.25, s=5, label=f"task {t} {label}",
                       rasterized=True)
    ax.set_title(f"Blocks {w_start}–{w_end - 1}", fontsize=9, pad=3)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_xlabel("PC1", fontsize=7, labelpad=1)
    ax.set_ylabel("PC2", fontsize=7, labelpad=1)

for idx in range(n_windows, len(axes)):
    axes[idx].set_visible(False)

handles, lbls = axes[0].get_legend_handles_labels()
fig.legend(handles, lbls, loc="lower right", fontsize=9, markerscale=2,
           title=f"tasks {POOL_TASKS}", title_fontsize=9, ncol=len(POOL_TASKS))

total_pairs = sum(min(len(activations_by_task[t]["positive"]),
                      len(activations_by_task[t]["negative"])) for t in POOL_TASKS)
fig.suptitle(
    f"Layer-pooled PCA  (window K={K} blocks, {n_windows} windows) — PC1 vs PC2\n"
    f"Tasks {POOL_TASKS}  ·  {total_pairs} contrastive pairs  ·  color=condition  shape=task\n"
    f"each point = one (observation × block) projection",
    fontsize=10,
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
out_path = Path(f"{OUTPUT_DIR}/cosmos_layer_pooled_k{K}.png")
out_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_path, dpi=150, bbox_inches="tight")
# plt.show()
print(f"Saved → {out_path}")